# Features de texto por dicionário e regra como covariável de vigilância

Este notebook abre o **braço 01** do projeto: a representação mais conservadora
de uma notícia — dicionário, regra e contagem —, sem modelo, sem GPU e sem
treino. Ele substitui a versão anterior ("News Pipeline — Step by Step"), que
percorria o mesmo caminho em inglês e sem citar de onde vinha cada escolha. Aqui
cada decisão de desenho tem seção própria e um bloco de **Aterramento** que diz de
que trabalho ela vem e com que número.

O caminho é: `manifest` + HTML → extração por geração de template → `ew` (o
domingo que começa a semana epidemiológica) → corpus congelado → sinais por artigo
→ features semanais por localidade. Ele espelha `src/pipeline/news/` exatamente:
desenrola `build()` para que cada transformação fique visível, e roda sobre os
fixtures versionados em `tests/fixtures/news/` — cinco páginas salvas, uma por
caminho de extração — para executar em segundos. As seções marcadas **Real** medem
o corpus de verdade (4.569 artigos, 2002–2026) quando os artefatos estão em disco.

**Pipeline:** manifest + HTML → extração por era → `ew` → corpus congelado →
sinais por artigo → features (ew, location) → junção opcional com o SINAN

## Onde este braço fica

O projeto compara **três representações concorrentes de um mesmo corpus de
notícia** — três modos de virar o mesmo artigo em feature de modelo — e pergunta
qual prevê melhor a dengue. Cada representação é um **braço**:

| Braço | Representação | Notebook | Estado |
|---|---|---|---|
| **01** | léxico PT-BR + regras escritas à mão | **este** | implementado |
| 02 | embeddings densos, agregados por semana | `news_embeddings_pipeline.ipynb` | implementado (bracket de 6) |
| 03 | modelagem de tópicos temporal (DMR) | `news_topics_pipeline.ipynb` | implementado |

Para a comparação valer, tudo menos a representação fica fixo: mesmo corpus, mesmo
calendário, mesmo alvo, mesmo modelo, mesmas métricas. Essas peças fixas estão
escritas em [`notes/00-protocolo-comparacao.md`](../notes/00-protocolo-comparacao.md),
o **protocolo 00**. O estado canônico deste braço — hipótese, medições e
pendências — está em [`notes/01-lexico-regras.md`](../notes/01-lexico-regras.md).

## As cinco decisões

| # | Decisão | Escolha | Por quê |
|---|---|---|---|
| 1 | Representação | dicionário + regra, sem modelo | é a linhagem canônica da vigilância por notícia e a que se fecha primeiro; é a única que existe sem corpus anotado |
| 2 | Unidade | uma linha por (semana, localidade), localidade ∈ {BR} ∪ 27 UFs | é o grão que todo braço publica, para os três entrarem no mesmo modelo sem remanejar |
| 3 | Extração | duas gerações de template, uma regra cada | a Agência Brasil trocou de página em ~2014 e as duas gerações estão no corpus |
| 4 | Contagens | **máximo**, não soma | as contagens do corpus são acumuladas até a data, então somar escalaria com quantas vezes a imprensa repetiu o mesmo número |
| 5 | Contagens frágeis | `news_cases`/`news_deaths` publicados, com o piso medido | o slot mais valioso é o mais raro; o notebook mede isso em vez de supor |

## Como ler as referências

As fontes desta família estão fichadas em
[`articles/features-texto/`](../articles/features-texto/): `relatorio.md` (o estado da arte),
`ranking.md` (as abordagens ranqueadas) e os 35 PDFs. Ao longo do texto, cada
bloco **Aterramento** nomeia o trabalho, o número que sustenta a escolha e o
arquivo local.

Um aviso de transferência, o mesmo que vale para o braço 02: **quase toda a
evidência de extração por dicionário é medida em inglês, sobre doenças de alta
noticiabilidade e, muitas vezes, sem baseline.** O que transfere para cá é o
mecanismo — a cascata manchete→corpo, o portão de contagem, o grão
doença–localidade —, não o número. O português aparece na literatura como
ausência, não como resultado, e é exatamente onde este braço trabalha.


## O que este notebook tem que provar

Sete afirmações, cada uma medida abaixo por um caminho que não é o do artefato
publicado. Elas são o contrato do notebook: o que não estiver aqui não é
resultado.

1. As duas gerações de template extraem um **corpo real** — era B de
   `div.inside`, era C de `div.node-conteudo` — e nenhum corpo era-B é a barra
   "Últimas notícias".
2. `ew` é sempre um **domingo**, e é o mesmo calendário em que o SINAN está
   indexado.
3. `char_count` é `len(text)`, e o artigo real mais curto passa de `MIN_CHARS`.
4. Todo artigo contribui com uma linha `BR`; só alguns também nomeiam uma UF.
5. Semana sem artigo é **ausente**, não zero.
6. `news_cases`/`news_deaths` são **máximos**, não somas, e são nulos na maioria
   das semanas.
7. O fixture tem cinco páginas: não mostra a fatia em espanhol, os portões de
   contagem em escala, nem o viés por UF. Isso é medido nas seções **Real**.


## Setup

Dois modos, e a diferença importa. **Fixture** roda sobre as cinco páginas
versionadas em `tests/fixtures/news/` e não precisa de rede nem de artefato
construído — é o que garante que este notebook executa do zero. **Real** mede os
parquets publicados em `data/pipeline/news/`, quando existem.

O princípio é o do braço 02: um resultado que não pode ser reproduzido sem rede
não é um resultado. Por isso as células de fixture rodam sempre e as de medida
real se anunciam como tais.


In [ ]:
import sys
from pathlib import Path

import polars as pl

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from pipeline.news import corpus as news_corpus
from pipeline.news.common import strip_accents
from pipeline.news.text.spec import (
    ARTICLE_ID, CHAR_COUNT, ERA, EW, EXTRACTOR, LANG_GUESS, MIN_CHARS,
    TEXT, TITLE,
)
from pipeline.news.text.spec import OUTPUT_COLUMNS as CORPUS_COLUMNS
from pipeline.news.text import steps as text_steps
from pipeline.news.text.integrity import run_all as corpus_checks
from pipeline.news.lexicon.spec import (
    LOCATION, NEWS_ARTICLES_WITH_CASES, NEWS_CASES, NEWS_DEATHS,
    NEWS_SEVERITY, NEWS_VOLUME,
)
from pipeline.news.lexicon.spec import OUTPUT_COLUMNS as FEATURE_COLUMNS
from pipeline.news.lexicon import steps as lex_steps
from pipeline.news.lexicon.integrity import run_all as feature_checks
from pipeline.sinan.cases import weekly_cases

pl.Config.set_tbl_rows(14)
pl.Config.set_fmt_str_lengths(70)

FIXTURES = PROJECT_ROOT / "tests" / "fixtures" / "news"
LEXICON_PATH = PROJECT_ROOT / "data" / "news" / "reference" / "lexico_dengue.yaml"
FU_PATH = PROJECT_ROOT / "data" / "epidemiological" / "br_federative_units.csv"

CORPUS_PATH = PROJECT_ROOT / "data" / "pipeline" / "news" / "EBC_dengue_text.parquet"
FEATURES_PATH = PROJECT_ROOT / "data" / "pipeline" / "news" / "EBC_dengue_lexicon_EW.parquet"
SINAN_PATH = PROJECT_ROOT / "data" / "epidemiological" / "SINAN" / "SINAN_dengue_cases.parquet"

REAL_CORPUS = CORPUS_PATH.exists()
REAL_FEATURES = FEATURES_PATH.exists()

print("modo: fixture (cinco páginas) + real" if REAL_CORPUS else "modo: FIXTURE — as seções Real ficam vazias")
print(f"corpus publicado: {'sim' if REAL_CORPUS else 'não'}   "
      f"features publicadas: {'sim' if REAL_FEATURES else 'não'}")


## 1. Por que dicionário e regra, e não um modelo

Esta escolha não vem de um benchmark — nenhuma das três representações foi
escolhida por ganhar de outra —, vem de um conjunto de restrições declaradas.
Cada linha da tabela é uma restrição que este braço tem que satisfazer para
existir no desenho do protocolo 00.

| Restrição | Peso | Quem passa |
|---|---|---|
| Rodar em CPU, em minutos, sem treino | alto | dicionário + regra |
| Existir sem corpus anotado | alto | dicionário + regra (o léxico é curadoria, não rótulo) |
| Ser auditável termo a termo | médio | dicionário + regra (o léxico é YAML versionado) |
| Ter precedente em notícia de doença | alto | dicionário + regra (a linhagem inteira) |
| Ter precedente em **dengue** | médio | parcial — só em volume de busca, e em inglês |

A última linha é a honesta: a dengue aparece na literatura de vigilância por
notícia sobretudo por outra modalidade de dado — o volume de consultas em
buscador, não o texto da matéria (Milinovich et al., 2013; Madoff et al., 2011) —
ou como um caso entre vários (Barboza et al., 2014). Não há precedente de
extração por dicionário sobre notícia de dengue em português.


**Aterramento.** A linhagem é a vigilância baseada em eventos, toda de dicionário
mais regra. O HealthMap casa n-gramas de 1 a 10 palavras contra um gazetteer de
cerca de 2.300 padrões de local e 1.100 de doença, numa cascata que vai da
manchete ao corpo e para assim que encontra local e doença (Freifeld et al.,
2008, [01_freifeld_2007_healthmap_global_media_monitoring.pdf](../articles/features-texto/01_freifeld_2007_healthmap_global_media_monitoring.pdf)). O
BioCaster troca o casamento puro por reconhecimento de entidades sobre uma
ontologia de 102 doenças em 8 línguas (Collier et al., 2008,
[03_collier_2008_biocaster_text_mining_rumors.pdf](../articles/features-texto/03_collier_2008_biocaster_text_mining_rumors.pdf)). O PADI-web soma
vocabulário curado por especialistas, o gazetteer GeoNames, regras de data e
expressões regulares para contagens (Valentin et al., 2020,
[05_valentin_2020_padi_web_multilingual_ebs.pdf](../articles/features-texto/05_valentin_2020_padi_web_multilingual_ebs.pdf)). O que todos entregam ao
fim é um punhado de rótulos discretos, não um vetor: do texto sobrevivem doença e
local, e é isso que alimenta a contagem (Collier, 2011,
[08_collier_2011_cross_lingual_alerting_epidemic.pdf](../articles/features-texto/08_collier_2011_cross_lingual_alerting_epidemic.pdf)).

**A lacuna que justifica o trabalho.** O português está ausente dessa linha. O
HealthMap listava o português como "em desenvolvimento" em 2008 (Brownstein et
al., 2008, [02_brownstein_2008_surveillance_sans_frontieres_healthmap.pdf](../articles/features-texto/02_brownstein_2008_surveillance_sans_frontieres_healthmap.pdf));
o BioCaster não detectou a epidemia de dengue no Ceará em 2008 por falta de
cobertura em português (Collier, 2010,
[09_collier_2010_unusual_online_outbreak_news.pdf](../articles/features-texto/09_collier_2010_unusual_online_outbreak_news.pdf)); o survey de
geoparsing avalia 27 abordagens de reconhecimento de topônimo sem nenhum modelo
de português (Hu et al., 2023, [33_hu_2023_location_reference_recognition.pdf](../articles/features-texto/33_hu_2023_location_reference_recognition.pdf)).
Nenhum trabalho levantado mede extração por dicionário sobre notícia em português,
e nenhum mede atributos de superfície em português (Crossley et al., 2011,
[24_crossley_2011_readability_formulas_simplification.pdf](../articles/features-texto/24_crossley_2011_readability_formulas_simplification.pdf)). O recorte deste
projeto — dengue, Brasil, texto em português, agregação semanal por estado —
aparece na literatura como ausência.


## 2. O corpus e o conjunto de artigos

O corpus congelado é o **chão comum** do protocolo 00: os três braços leem a mesma
tabela, ou a comparação mede o extrator de texto em vez da representação. Ele é
produzido uma vez (`arboili transform news-text`) e não é mexido depois.

Sobre esse corpus incidem **dois portões**, e eles são uma função só —
`news_corpus.select_articles`, importada aqui e pelos braços 02 e 03:

- **idioma**: mantém `lang_guess == "pt"`; a Agência Brasil publica edição em
  espanhol, e um léxico PT-BR subconta exatamente essas semanas;
- **termo de doença**: mantém artigos que nomeiam uma doença da família; o portão
  derruba páginas de errata, notas de agenda e digests cujo vínculo com dengue é
  incidental.

Reimplementar esse filtro aqui mediria o filtro, não a representação — por isso
ele é chamado, nunca reescrito.


In [ ]:
manifest = text_steps.load_manifest(FIXTURES)
print(f"{len(manifest)} linhas de manifest carregam um artigo salvo")

rows = list(text_steps.iter_rows(FIXTURES))
{ARTICLE_ID: rows[0][ARTICLE_ID], ERA: rows[0][ERA], CHAR_COUNT: rows[0][CHAR_COUNT]}


## 3. Duas gerações de template, e a armadilha da barra

O site foi redesenhado por volta de 2014 e as duas gerações estão no corpus. A
era B põe o corpo em `div.inside` e o *nome do site* no `<h1>`; a era C usa
`div.node-conteudo` e o título de verdade no `<h1>`. Confiar no `<h1>` teria
rotulado 2.628 artigos como "Agência Brasil".

`div.inside` casa **três** elementos por página da era B — um cabeçalho de 20
caracteres, o artigo e uma barra de notícias —, então o extrator pontua os
candidatos em vez de pegar o primeiro. A pontuação é `(texto de parágrafo, texto
sem link)`: um artigo tem `<p>`, uma barra não.


In [ ]:
corpus = text_steps.build(FIXTURES)
assert tuple(corpus.columns) == CORPUS_COLUMNS
corpus.group_by(ERA, EXTRACTOR).len().sort(ERA, EXTRACTOR)


### A armadilha, mostrada

Este é o mais perigoso dos dois bugs de extração que o projeto nomeia. O **bug 1**
pegava um cabeçalho de página de 20 caracteres como corpo; o **bug 2** pegava a
barra lateral "Últimas notícias" no lugar do artigo.

O bug 2 é o que poderia ter passado em silêncio. Pontuar os candidatos pelo texto
*total* escolhia a barra em **1.097 dos 2.628 artigos da era B (42%)** — 1.086
caracteres de texto de link contra um artigo de 677, longo o bastante para passar
em todo check de comprimento. A correção pontua pelo texto de parágrafo primeiro,
que a barra não tem.

As três linhas era-B do fixture abaixo são as mais curtas do corpus, que é onde
uma regressão apareceria. A primeira é uma **página sem corpo**: a coleta salvou a
manchete e pulou direto para a barra, então só sobrou a assinatura.


In [ ]:
(
    corpus.filter(pl.col(ERA) == "b")
    .sort(CHAR_COUNT)
    .select(ARTICLE_ID, EXTRACTOR, CHAR_COUNT,
            pl.col(TEXT).str.slice(0, 70).alias("text_head"))
)


**Aterramento.** Tratar a qualidade do texto extraído como etapa, e não como
detalhe, vem da construção de corpus de notícia geolocalizado: Kriesch & Losacker
(2025) descartam documentos por sete heurísticas de qualidade — número mínimo de
frases, fração de palavras não alfabéticas, palavras por linha, contagem de
palavras entre 50 e 10.000 e remoção de duplicatas (Kriesch & Losacker, 2025,
[29_kriesch_2025_geolocated_german_news_dataset.pdf](../articles/features-texto/29_kriesch_2025_geolocated_german_news_dataset.pdf)). A barra "Últimas
notícias" é o caso que essas heurísticas pegariam e o check de tamanho não pega —
1.086 caracteres de link passam em qualquer piso de comprimento. O que o corpus
deste projeto faz a mais é ancorar a página sem corpo na assinatura final, e não
na mera brevidade: o bug 1 também produzia corpos de 20 caracteres, então uma
regra só de tamanho reclassificaria aquela regressão como coleta truncada e a
isentaria do piso.


## 4. `ew` — o domingo que começa a semana

`add_ew` lê `published_at` e arredonda para baixo até o domingo com o helper
compartilhado `pipeline.epiweek.floor_to_sunday` — nunca `dt.truncate("1w")`, que
o polars ancora na segunda. SINAN, Google Trends e clima usam o mesmo helper, de
modo que o eixo de junção é uma definição, e não quatro.


In [ ]:
dated = text_steps.add_ew(pl.DataFrame(rows, schema=text_steps.SCHEMA))
assert dated.filter(pl.col(EW).dt.weekday() != 7).is_empty(), "nem tudo é domingo"
dated.select(ARTICLE_ID, "published_at", EW).sort(EW)


## 5. `char_count` e o piso de comprimento

`char_count` é `len(text)`, guardado para que um consumidor possa filtrar sem
tocar no corpo. `MIN_CHARS` é 100: uma extração correta chega no fundo a 128
caracteres (uma nota de 2002 de uma frase), enquanto uma regressão de cabeçalho
rende cerca de 20.

`bodyless_expr` separa os dois casos. Ele se ancora no crédito de site que fecha a
assinatura, e não na brevidade sozinha — e é essa âncora que faz a regra
funcionar. Se fosse só comprimento, o bug 1 seria reclassificado como coleta
truncada e isento de `min_length`; o check pararia de checar.


In [ ]:
(
    corpus.with_columns(text_steps.bodyless_expr().alias("bodyless"))
    .select(ERA, CHAR_COUNT, "bodyless",
            pl.col(TEXT).str.slice(0, 45).alias("text_head"))
    .sort(CHAR_COUNT)
)


## 6. `build()` encadeia: o corpus congelado

`build()` = linhas → schema declarado → `add_ew` → `order_columns`. A ordem de
colunas publicada é fixa, e o schema é declarado em vez de inferido para que uma
coluna toda nula ainda receba o dtype publicado. Chaves primeiro, depois
proveniência, depois a carga — a mesma regra de ordenação do braço 02.


In [ ]:
corpus.select(ARTICLE_ID, EW, ERA, EXTRACTOR, CHAR_COUNT, LANG_GUESS)


### Verificação — corpus

Os mesmos dez checks que o CLI e a suíte de testes rodam. Dois deles **reportam em
vez de falhar**, e a distinção é deliberada: `langs_known` e `bodyless_pages`
descrevem propriedades da *fonte* (a Agência Brasil publica em espanhol; três
páginas foram coletadas sem corpo), enquanto `min_length` e `title_agreement`
descrevem defeitos *desta etapa* e falham.


In [ ]:
for check in corpus_checks(corpus):
    print(check)


## 7. Braço 01: o léxico e o casador de UF

O léxico é YAML versionado. Os termos são guardados sem acento e casados contra
uma cópia normalizada do texto, então `obito` casa `óbito`; termos de uma palavra
ganham plural gerado (`-cao → -coes`, `-r/-z → +es`). A severidade carrega pesos
ordinais — `alerta` 1, `surto` 2, `epidemia` 3.

Nomes completos de estado casam **sensível a maiúsculas, sobre o texto original**,
e isso pesa mais do que parece: dobrar acentos faria `Pará` e a preposição `para`
serem a mesma string, escopando quase todo artigo para o Pará. Siglas de duas
letras exigem pista locativa (`no estado de SP`), porque `SE` é pronome reflexivo e
`PA` colide com `p.a.`.


In [ ]:
lexicon = lex_steps.load_lexicon(LEXICON_PATH)
matcher = lex_steps.load_uf_matcher(FU_PATH)
{
    "version": lexicon.version,
    "severity_tiers": {tier.name: tier.weight for tier in lexicon.severity},
    "disease_terms": len(lexicon.disease),
    "clinical_terms": len(lexicon.clinical),
    "vector_terms": len(lexicon.vector),
    "uf_name_patterns": len(matcher.name_patterns),
    "require_disease_term": lexicon.require_disease_term,
}


**Aterramento.** Manter o léxico à mão é condição do método, não acessório dele. O
HealthMap precisou de um especialista de idioma e de entrada manual contínua —
atualizaram o léxico quando Sérvia e Montenegro se separaram (Freifeld et al.,
2008, [01_freifeld_2007_healthmap_global_media_monitoring.pdf](../articles/features-texto/01_freifeld_2007_healthmap_global_media_monitoring.pdf)); o PADI-web
depende de vocabulário curado por especialistas mais o gazetteer GeoNames
(Valentin et al., 2020, [05_valentin_2020_padi_web_multilingual_ebs.pdf](../articles/features-texto/05_valentin_2020_padi_web_multilingual_ebs.pdf)); a
base de notícia de surto da OMS foi codificada manualmente em 2.789 relatos, e os
autores registram a inconsistência dos documentos-fonte — a mesma doença sob cinco
nomes, cidades grafadas de formas variadas, relatos duplicados em dias
consecutivos (Carlson et al., 2023,
[18_carlson_2023_who_disease_outbreak_news_database.pdf](../articles/features-texto/18_carlson_2023_who_disease_outbreak_news_database.pdf)). É esse tipo de
variação que a normalização de acento e a regra de plural deste léxico absorvem.

O grão de localidade também tem lastro: unidades administrativas grossas — o nível
de país e estado — são a categoria mais fácil de reconhecer no survey de
geoparsing, com 13 de 27 métodos passando de 60% (Hu et al., 2023,
[33_hu_2023_location_reference_recognition.pdf](../articles/features-texto/33_hu_2023_location_reference_recognition.pdf)). Nossas 28 localidades são
exatamente esse nível. O que fica de fora é declarado: **municípios não são
resolvidos** para a UF, então um artigo que diz só "Araçutuba" contribui para BR e
não para SP.


## 8. Sinais por artigo

`analyse` funde título e corpo, dobra acentos uma vez e devolve o escore de
severidade, a maior contagem de casos e de óbitos que sobreviveu aos portões, se
há termo de doença, e quais UFs foram nomeadas.

No fixture toda severidade é **0**: nenhuma das cinco páginas salvas usa "surto",
"epidemia" ou "alerta". Isso é propriedade da amostra, não do escore — o corpus
real é onde a severidade tem sinal.


In [ ]:
for row in corpus.filter(pl.col(CHAR_COUNT) > 100).sort(EW).iter_rows(named=True):
    signals = lex_steps.analyse(row[TEXT], row[TITLE], lexicon, matcher)
    print(
        f"{row[ARTICLE_ID][:34]:34}  severity={signals.severity:2}"
        f"  cases={signals.cases}  deaths={signals.deaths}"
        f"  ufs={sorted(signals.locations)}"
    )


## 9. Agregação para (ew, location)

Só pares observados são emitidos. Semana sem artigo é **ausente**, não zero:
emitir a grade densa afirmaria "cobertura zero" para toda UF que a notícia
nacional simplesmente nunca nomeou, o que é uma afirmação diferente de "nenhuma
cobertura nesta semana". O consumidor faz left join sobre o calendário do alvo.

`news_volume` é contagem, `news_severity` é soma, e os dois slots de contagem são
**máximo**.


In [ ]:
features = lex_steps.build(corpus, lexicon, matcher)
assert tuple(features.columns) == FEATURE_COLUMNS
features


### Verificação — features

Oito checks: schema, chaves únicas, domingos, `locations ⊆ {BR} ∪ 27 UFs`,
contagens não negativas, o domínio de casos, `articles_with_cases ≤ volume`, e
**uma linha BR em toda semana**. Todo artigo contribui para BR por construção,
então uma semana sem BR significa que a distribuição por localidade derrubou algo.

No fixture, `locations_known` falha: cinco artigos nomeiam só RJ, então 26 UFs
nunca aparecem. Na saída real, as 27 passam.


In [ ]:
for check in feature_checks(features, FU_PATH):
    print(check)


## 10. O slot mais frágil: a contagem de casos

Se a extração por dicionário devolve poucos campos, o mais frágil deles merece
seção própria, porque é dele que dependeria uma variável de casos construída a
partir da imprensa. A literatura é clara e desconfortável nesse ponto.


**Aterramento.** O número de casos é o campo de pior desempenho onde é medido, e é
declarado inexistente em produção. O pipeline que usa três modelos de linguagem
para ler relatórios oficiais da OMS alcança F1 de 0,962 para país, 0,869 para
data, 0,851 para doença — e **0,658 para número de casos**, o pior dos campos; a
extração de óbitos nem foi avaliada, por não estar anotada no conjunto de
referência (Consoli et al., 2025,
[19_consoli_2025_knowledge_graph_who_don.pdf](../articles/features-texto/19_consoli_2025_knowledge_graph_who_don.pdf)). Na avaliação do PADI-web
sobre 352 notícias, o número de casos fica em 85%, abaixo dos 95% de doença e
hospedeiro (Arsevska et al., 2018,
[15_arsevska_2018_web_monitoring_animal_diseases.pdf](../articles/features-texto/15_arsevska_2018_web_monitoring_animal_diseases.pdf)). E a revisão das
tarefas de extração registra que nenhum algoritmo de PLN incorporado a sistemas de
vigilância em produção era capaz de extrair contagem de casos — havia só
protótipos dos próprios autores (Ng et al., 2020,
[10_ng_2020_nlp_extract_news_event_based.pdf](../articles/features-texto/10_ng_2020_nlp_extract_news_event_based.pdf)).

A consequência para este braço é uma expectativa calibrada, não uma promessa: se o
slot mais valioso é o mais frágil na literatura, ele deve ser o mais raro aqui
também. A seção Real abaixo mede quanto.


### Real — quanto o slot aparece

Esta é a medição que responde à pergunta aberta de
[`notes/01-lexico-regras.md`](../notes/01-lexico-regras.md) sobre se os dois slots
de contagem valem o esforço. Olhe a linha `null_count`: `news_volume` e
`news_severity` estão em **todas** as semanas, `news_cases` em cerca de metade, e
`news_deaths` em um quinto. É por isso que a nota encaminha a extração estruturada
de contagem para o braço 04 (extração com LLM) em vez de deixá-la na regex.


In [ ]:
if REAL_FEATURES:
    real_features = pl.read_parquet(FEATURES_PATH)
    national = real_features.filter(pl.col(LOCATION) == "BR")
    print(f"linhas={real_features.height:,}  semanas={real_features[EW].n_unique():,}"
          f"  localidades={real_features[LOCATION].n_unique()}\n")
    print(national.select(NEWS_VOLUME, NEWS_SEVERITY, NEWS_CASES, NEWS_DEATHS).describe())
    print()
    print(
        real_features.filter(pl.col(LOCATION) != "BR")
        .group_by(LOCATION).len().sort("len", descending=True).head(5)
        .rename({"len": "semanas_cobertas"})
    )
    print("semanas cobertas por UF — não artigos: um artigo que nomeia dois estados"
          " contribui para cada um, e as contagens de artigo do README (RJ 1.636"
          " vs SP 863) são outra medida")
else:
    print("Não construído — rode: uv run arboili transform news-lexicon")


## 11. Os três portões nas contagens

Extrair número de prosa jornalística é dominado por números que não são da dengue.
Uma contagem só é mantida se:

1. **não for negada** — nenhuma pista de negação nos 40 caracteres anteriores;
2. **não for qualificada por outra doença** — `11.881.430 casos de DRSAI` não é
   nossa;
3. **estiver numa frase que nomeia doença da família dengue**.

O portão 3 é o mais estrito e o que custa revocação: `No estado de São Paulo,
foram 36.791 casos` é descartado quando a frase nunca diz "dengue". A direção é
deliberada — uma contagem uma ordem de grandeza errada é pior para o modelo do que
uma contagem faltante.

Duas regras não óbvias seguem a sonda. `mil` só multiplica quando o número **não**
já vem pontuado (`36.791 mil casos` é 36.791, não 36,8 milhões). E a severidade é
ordinal: a última linha da sonda marca 6, porque "alerta" + "surto" + "epidemia"
são 1 + 2 + 3.


In [ ]:
probe = [
    "São Paulo registrou 36.791 casos de dengue em 2024.",
    "No estado de São Paulo, foram 36.791 casos no ano passado.",
    "Não houve 500 casos de dengue na cidade.",
    "Foram 11.881.430 casos de DRSAI no país.",
    "O país somou 700 mil mortes por covid, segundo estudo sobre dengue.",
    "Dengue: 500 mil casos prováveis e 312 óbitos.",
    "Dengue: 36.791 mil casos.",
    "Alerta para o surto de dengue; pode haver epidemia.",
]
for line in probe:
    normalized = strip_accents(line)
    cases, deaths = lex_steps.count_mentions(normalized, lexicon)
    severity = lex_steps.severity_score(normalized, lexicon)
    print(f"{line[:56]:58} cases={str(cases):7} deaths={str(deaths):5} severity={severity}")


**Aterramento.** O portão existe porque a discordância humana se concentra
exatamente nos campos de que depende uma série por localidade. Dois anotadores
marcando 100 notícias concordaram no **número de eventos** em apenas 67% dos
documentos (Conway et al., 2010,
[30_conway_2010_disease_outbreak_event_corpus.pdf](../articles/features-texto/30_conway_2010_disease_outbreak_event_corpus.pdf)); no esquema
espaço-temporal, o acordo para o local exato cai a 0,806 e 0,749 e sobe a 0,997
quando se aceita concordância aproximada (Chanlekha et al., 2010,
[31_chanlekha_2010_spatial_temporal_granularity_reports.pdf](../articles/features-texto/31_chanlekha_2010_spatial_temporal_granularity_reports.pdf)). Se nem dois
humanos concordam no número, um casador de numeral vai errar, e o portão é a
forma barata de errar para o lado da falta. Os documentos-fonte são internamente
inconsistentes — datas impossíveis, duplicatas em dias consecutivos (Carlson et
al., 2023, [18_carlson_2023_who_disease_outbreak_news_database.pdf](../articles/features-texto/18_carlson_2023_who_disease_outbreak_news_database.pdf)) —, o
que só reforça que a contagem crua não é confiável sem contexto.


## 12. A armadilha da contagem acumulada

A imprensa repete números **acumulados** em vez de incrementos semanais: a maioria
dos artigos diz "ao longo de 2024, 6,6 milhões de casos". O máximo semanal é,
portanto, a *escala do maior número que a imprensa citou naquela semana*, não a
incidência da semana. Usar isso como covariável semanal sem modelar a diferença é
um erro real — e é um item aberto do protocolo 00.

O que a literatura diz sobre isso é o mais importante desta seção: **ela não
enfrenta o problema.** Nenhum dos sistemas que extraem número de casos (Valentin
et al., 2020; Valentin et al., 2021; Arsevska et al., 2018) descreve um
procedimento para distinguir uma contagem acumulada de uma nova, para datar a
contagem ou para convertê-la em incremento. A tarefa é enunciada como localizar o
numeral e a doença a que ele se refere, sem discussão sobre a semântica temporal
do número. A resposta operacional da área foi contornar: contar **relatos** em vez
de casos.


In [ ]:
if REAL_FEATURES:
    nacional = real_features.filter(pl.col(LOCATION) == "BR")
    extremos = (
        nacional.filter(pl.col(NEWS_CASES).is_not_null())
        .sort(NEWS_CASES, descending=True)
        .select(EW, NEWS_CASES, NEWS_DEATHS)
        .head(5)
    )
    print(extremos)
    print("\nO máximo de news_cases é o recorde real de 2024. O máximo de"
          " news_deaths NÃO é dengue — é uma estimativa global de doenças"
          " negligenciadas numa frase que por acaso nomeia dengue, e sobrevive aos"
          " três portões. Resíduo documentado, não suavizado.")
else:
    print("Não construído — rode: uv run arboili transform news-lexicon")


## 13. Volume mede atenção, não incidência

Este é o achado transversal mais desconfortável da literatura, e ele é uma
restrição de desenho para os três braços, não um detalhe lateral: a frequência de
notícia não acompanha a carga de doença.


**Aterramento.** O HealthMap observa que a frequência de relato segue a disrupção
econômica e social percebida, não a morbidade ou mortalidade (Brownstein et al.,
2008, [02_brownstein_2008_surveillance_sans_frontieres_healthmap.pdf](../articles/features-texto/02_brownstein_2008_surveillance_sans_frontieres_healthmap.pdf)). O
BioCaster mostra que a contagem bruta reflete a preocupação do leitor e carrega um
forte efeito de dia da semana — 1,37 documento por dia em dias úteis contra 0,49
em fins de semana (Collier, 2010,
[09_collier_2010_unusual_online_outbreak_news.pdf](../articles/features-texto/09_collier_2010_unusual_online_outbreak_news.pdf)). A revisão da vigilância
por volume de busca documenta o caso extremo: o interesse gerado pela gripe em
2009 obscureceu a atividade real de todos os outros anos (Milinovich et al., 2013,
[17_milinovich_2013_internet_based_surveillance_review.pdf](../articles/features-texto/17_milinovich_2013_internet_based_surveillance_review.pdf)), e picos de
atenção midiática produzem aumento de sinal sem aumento de doença (Madoff et al.,
2011, [20_madoff_2011_monitoring_dengue_activity.pdf](../articles/features-texto/20_madoff_2011_monitoring_dengue_activity.pdf)). A avaliação contra o
FluNet é o número que calibra a expectativa: os sistemas detectaram 73,5% dos
surtos em algum momento, mas só **9,2% dentro de ±2 semanas** do início (Ganser et
al., 2022, [11_ganser_2022_global_variations_event_based.pdf](../articles/features-texto/11_ganser_2022_global_variations_event_based.pdf)).

Para uma variável construída a partir de notícia, a consequência é direta: o
volume pode estar medindo atenção pública. É por isso que a leitura descritiva
abaixo é chamada de descritiva, e não de resultado.


## 14. Leitura descritiva (2014–2026)

**Esta não é a comparação do protocolo 00.** É uma primeira olhada na pergunta do
artigo, e ela deve ser lida com três ressalvas. `news_volume` é a feature
defensável; `news_cases` é uma figura **acumulada** que a imprensa repete, então
correlacioná-la com incidência semanal mede outra coisa — está na tabela para
mostrar isso, não para ser usada. A cobertura por UF é esparsa: alguns estados são
nomeados em só 32 das 479 semanas que as duas séries compartilham, o que faz o
coeficiente deles ser ruidoso. E a série de notícia é nacional enquanto o alvo é
UF, então uma correlação por UF é na verdade "a cobertura nacional anda com a curva
deste estado?".

O SINAN aqui é `ew_symptom_onset` — o mesmo calendário de domingo — somado para
(semana, estado).


In [ ]:
# O alvo vem da definição compartilhada do protocolo 00: casos **suspeitos**, não
# confirmados. Somar `case_count` sem filtro arrastaria os casos descartados
# depois de investigação.
sinan_weekly = weekly_cases(pl.read_parquet(SINAN_PATH), by_state=True).rename(
    {"case_count": "casos"}
)

if REAL_FEATURES:
    br = (
        real_features.filter(pl.col(LOCATION) == "BR")
        .join(
            sinan_weekly.group_by("ew_symptom_onset")
            .agg(pl.col("casos").sum())
            .rename({"ew_symptom_onset": EW}),
            on=EW,
            how="inner",
        )
        .sort(EW)
    )
    print(f"semanas nacionais com as duas séries: {br.height:,}  "
          f"({br[EW].min()} → {br[EW].max()})\n")
    print(
        pl.DataFrame({
            "feature": [NEWS_VOLUME, NEWS_SEVERITY, NEWS_CASES],
            "spearman vs casos semanais": [
                br.select(pl.corr(f, "casos", method="spearman")).item()
                for f in (NEWS_VOLUME, NEWS_SEVERITY, NEWS_CASES)
            ],
        }).with_columns(pl.col("spearman vs casos semanais").round(3))
    )
else:
    print("Rode `uv run arboili transform news-lexicon` primeiro")


### Por UF

Cada estado ganha seu coeficiente, calculado só sobre as semanas em que o corpus
nomeia aquele estado. A coluna `semanas` é, portanto, o denominador honesto: um
coeficiente sobre 32 semanas é uma afirmação diferente de um sobre 265.


In [ ]:
if REAL_FEATURES:
    per_uf = (
        real_features.filter(pl.col(LOCATION) != "BR")
        .join(sinan_weekly, left_on=[EW, LOCATION],
              right_on=["ew_symptom_onset", "state_abbrev"], how="inner")
        .group_by(LOCATION)
        .agg(
            pl.len().alias("semanas"),
            pl.corr(NEWS_VOLUME, "casos", method="spearman").round(3).alias("rho_volume"),
            pl.corr(NEWS_SEVERITY, "casos", method="spearman").round(3).alias("rho_severity"),
            pl.corr(NEWS_CASES, "casos", method="spearman").round(3).alias("rho_cases"),
        )
        .sort("rho_volume", descending=True)
    )
    print(per_uf.head(8))
    print(f"...\n{per_uf.height} UFs, semanas por UF {per_uf['semanas'].min()}–{per_uf['semanas'].max()}")
    print(f"mediana rho_volume = {per_uf['rho_volume'].median():.3f}")


## 15. Antecedência ou atraso?

Se a cobertura midiática carregasse alerta precoce, `news_volume` na semana *t*
correlacionaria melhor com casos em *t+k* do que com casos em *t*. A tabela desloca
as duas séries e reporta o coeficiente de Spearman nacional em cada direção.


In [ ]:
if REAL_FEATURES:
    scan = []
    for k in range(0, 9):
        shifted = br.with_columns(
            pl.col("casos").shift(-k).alias("casos_futuros"),
            pl.col(NEWS_VOLUME).shift(-k).alias("volume_futuro"),
        )
        scan.append({
            "k (semanas)": k,
            "news[t] × casos[t+k]": shifted.select(
                pl.corr(NEWS_VOLUME, "casos_futuros", method="spearman")).item(),
            "casos[t] × news[t+k]": shifted.select(
                pl.corr("casos", "volume_futuro", method="spearman")).item(),
        })
    print(pl.DataFrame(scan).with_columns(pl.exclude("k (semanas)").round(3)))


## 16. O que este notebook encontrou

Os números abaixo são da execução que gerou este arquivo, não de uma expectativa.
Os de fixture valem para as cinco páginas salvas; os de medida real valem para o
corpus publicado.

**Do corpus congelado.** As duas gerações de template extraem corpo real, e o
piso de comprimento as separa de uma regressão de cabeçalho: o artigo real mais
curto tem 128 caracteres, um cabeçalho tem cerca de 20, e a página sem corpo é
identificada pela âncora da assinatura e não pelo tamanho. O corpus é nacional, e
a fatia em espanhol (171 de 4.569, 3,7%) é filtrada antes de qualquer contagem.

**Do braço 01.** `news_volume` e `news_severity` estão presentes em todas as 965
semanas; `news_cases` em 511 e `news_deaths` em 189. Só 24% dos artigos (1.089 de
4.569) declaram alguma contagem de casos, e 6% (276) uma de óbitos. O slot mais
valioso é o mais raro — exatamente o que a literatura de extração de contagem
previa.

**Da leitura descritiva.** O coeficiente nacional de `news_volume` contra casos é
**0,297** (severidade 0,270; casos 0,237) sobre as 479 semanas que as duas séries
compartilham, e ele **pica em `k = 0`**, caindo nos dois sentidos: a notícia anda
*com* a curva, não à frente dela. É o mesmo achado de Ganser et al. (2022), que
detectou 73,5% dos surtos mas só 9,2% dentro de ±2 semanas. Por UF, a mediana de
`rho_volume` é 0,221 e o `rho_cases` **vai de +0,70 a −0,63** porque correlaciona
um estoque acumulado com um fluxo semanal — está na tabela para que o leitor veja
falhar, e essa falha é o argumento para levar a extração estruturada de contagem
ao braço 04.


## 17. O que este notebook não prova

A lista é longa de propósito. Um notebook que só enuncia o que provou é um
notebook que escondeu as próprias bordas. Cada item diz onde o limite está escrito.

1. **Não é a comparação do protocolo 00.** Braços 01, 02 e 03 sob um mesmo modelo
   e um mesmo conjunto de métricas não rodaram; os coeficientes descritivos acima
   não podem ser citados como resultado dele. O protocolo está em
   [`notes/00-protocolo-comparacao.md`](../notes/00-protocolo-comparacao.md).
2. **Sem estudo de anotação.** Precisão e revocação por campo não são medidas aqui.
   Não existe corpus anotado de dengue em PT-BR, e construí-lo é a contribuição
   aberta de [`notes/01-lexico-regras.md`](../notes/01-lexico-regras.md).
3. **Sem correção de acumulado vs. semanal.** A imprensa repete números acumulados;
   o ajuste do lado do modelo é item aberto do protocolo 00, e a literatura não o
   trata (Valentin et al., 2020; Arsevska et al., 2018; Consoli et al., 2025).
4. **Sem normalização por UF.** O viés de menção não é estável no tempo — RJ lidera
   SP em 2002–2004 e a relação inverte a partir de 2015. Uma feature por UF é
   comparável à própria história antes de ser comparável entre UFs.
5. **A negação é janela de caracteres, não sintaxe.** "não apenas um surto" perde
   um "surto" real. A magnitude desse erro só a anotação quantifica.
6. **Municípios não são resolvidos.** Um artigo que diz só "Araçatuba" conta para
   BR e não para SP, enviesando o volume por UF para baixo, de forma desigual.
7. **`lang_guess` não detecta inglês.** É contagem de marcadores, não detecção de
   idioma; cerca de 180 artigos rotulados `pt` são inglês. O filtro os mantém.
8. **Uma única fonte, estatal e nacional.** O descompasso espacial entre notícia
   nacional e alvo estadual pertence à fonte, não a esta etapa — está declarado,
   não resolvido.
9. **O fixture é de cinco páginas.** Ele não mostra a fatia em espanhol, os portões
   de contagem em escala nem o viés por UF; isso é medido só nas seções **Real**.
10. **Três páginas não têm corpo.** Elas contam para `news_volume` e nada mais, e
    `check_bodyless_pages` as nomeia.


## 18. Como reproduzir

Este notebook espelha `transform()` em `src/pipeline/news/*/transform.py`:

```bash
uv run arboili ebc                        # coleta os HTMLs da Agência Brasil
uv run arboili transform news-text        # 4.569 artigos → corpus congelado  (~45 s)
uv run arboili transform news-lexicon     # corpus → features por UF × semana   (~6 s)
```

Escreve `data/pipeline/news/EBC_dengue_text.parquet` e
`..._lexicon_EW.parquet` (+ `.csv.gz`).

### Onde cada decisão está registrada

O mapa de rastreabilidade do notebook: cada linha aponta para o arquivo onde a
decisão está escrita, e não para uma paráfrase dela.

| Decisão | Onde está escrita |
|---|---|
| Representação (dicionário + regra) | [`notes/01-lexico-regras.md`](../notes/01-lexico-regras.md) · [`articles/features-texto/ranking.md`](../articles/features-texto/ranking.md) |
| O que é fixo entre os braços | [`notes/00-protocolo-comparacao.md`](../notes/00-protocolo-comparacao.md) |
| Colunas e constantes de casamento | [`src/pipeline/news/lexicon/spec.py`](../src/pipeline/news/lexicon/spec.py) |
| As cinco features e os portões | [`src/pipeline/news/README.md`](../src/pipeline/news/README.md) §2 |
| Extração por era, piso de comprimento | [`src/pipeline/news/text/spec.py`](../src/pipeline/news/text/spec.py) · [`common.py`](../src/pipeline/news/common.py) |
| Corpus congelado e os dois portões | [`src/pipeline/news/corpus.py`](../src/pipeline/news/corpus.py) |
| Limitações declaradas | [`src/pipeline/news/README.md`](../src/pipeline/news/README.md) §6 |
| Literatura e números citados | [`articles/features-texto/relatorio.md`](../articles/features-texto/relatorio.md) |


## Resumo executivo

O braço 01 transforma cada notícia em número por dicionário, regra e contagem — a
representação mais conservadora das três, e a única que existe sem corpus anotado.
Ele publica cinco features por (semana, localidade): `news_volume`,
`news_severity`, `news_cases`, `news_deaths` e `news_articles_with_cases`.

O que a execução encontrou: o corpus e a agregação se sustentam (as duas gerações
de template extraem corpo real, `ew` é domingo em todo lugar, toda semana tem uma
linha BR), mas os dois slots de contagem são magros — 24% dos artigos declaram
casos, metade das semanas fica sem `news_cases`. Isso não é defeito de
implementação: é o que a literatura de extração de contagem previa, e é a razão de
a extração estruturada pertencer ao braço 04.

O que este notebook **não** entrega é a comparação: nenhum número aqui é o
resultado do protocolo 00. As features estão prontas para ele, e as bordas — sem
anotação, sem correção de acumulado, sem normalização por UF — estão declaradas
acima, não escondidas.
